<a href="https://colab.research.google.com/github/Amika1118/DSGP_Group_38/blob/Market-Price-Prediction/Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%cd /content
!git clone https://github.com/Amika1118/DSGP_Group_38.git
%cd DSGP_Group_38

/content
Cloning into 'DSGP_Group_38'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 110 (delta 29), reused 59 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 6.38 MiB | 8.73 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/DSGP_Group_38


In [3]:
!git checkout Market-Price-Prediction

Branch 'Market-Price-Prediction' set up to track remote branch 'Market-Price-Prediction' from 'origin'.
Switched to a new branch 'Market-Price-Prediction'


In [4]:
!git config --global user.name "Lasani Layathma"
!git config --global user.email "lasani.20241357@iit.ac.lk"

In [5]:
from getpass import getpass
token = getpass("Enter GitHub token: ")
!git remote set-url origin https://{token}@github.com/Amika1118/DSGP_Group_38.git

Enter GitHub token: ··········


In [7]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Load the dataset
df = pd.read_csv('/content/retail_fuel_weekly_merged (1).csv')

# Display basic info to understand the data structure
print("Dataset shape:", df.shape)  # Shows number of rows and columns
print("\nColumn info:")
print(df.info())  # Shows data types and non-null counts
print("\nMissing values per column:")
print(df.isnull().sum())  # Counts NaN values in each column
print("\nUnique values in categorical columns:")
print("Variety:", df['Variety'].unique())  # Shows all vegetable types
print("Location:", df['Location'].unique())  # Shows all locations
print("Season:", df['Season'].unique())  # Shows all seasons
print("Price_Band:", df['Price_Band'].unique())  # Shows price categories

Dataset shape: (7784, 23)

Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7784 entries, 0 to 7783
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Year              7784 non-null   int64  
 1   Month             7784 non-null   int64  
 2   Week_Number       7784 non-null   int64  
 3   Start_Date        7784 non-null   object 
 4   End_Date          7784 non-null   object 
 5   Location          7784 non-null   object 
 6   Variety           7784 non-null   object 
 7   Weekly_Price      7784 non-null   float64
 8   Price_Std         7777 non-null   float64
 9   Days_In_Week      7784 non-null   int64  
 10  Month_Year        7784 non-null   object 
 11  Price_Change      7778 non-null   float64
 12  Price_Change_Pct  7778 non-null   float64
 13  Rolling_Mean_4w   7784 non-null   float64
 14  Rolling_Std_4w    7778 non-null   float64
 15  Rolling_Mean_13w  7784 non-null   float64
 16  Ro

In [8]:
# Create a copy to avoid modifying original data
df_preprocessed = df.copy()

# Find columns with missing values
missing_cols = df_preprocessed.columns[df_preprocessed.isnull().any()].tolist()
print("Columns with missing values:", missing_cols)

# Strategy 1: For Price_Std (price standard deviation), assume 0 if missing
# This means no price variation was recorded
df_preprocessed['Price_Std'] = df_preprocessed['Price_Std'].fillna(0)

# Strategy 2: For Price_Change and Price_Change_Pct in first week, use 0
# First week can't have a change from previous week
df_preprocessed['Price_Change'] = df_preprocessed['Price_Change'].fillna(0)
df_preprocessed['Price_Change_Pct'] = df_preprocessed['Price_Change_Pct'].fillna(0)

# Strategy 3: Handle rolling statistics (4-week, 13-week, 26-week averages)
rolling_cols = ['Rolling_Mean_4w', 'Rolling_Std_4w',
                'Rolling_Mean_13w', 'Rolling_Std_13w',
                'Rolling_Mean_26w', 'Rolling_Std_26w']

# For each vegetable type, handle the first week's rolling stats
for variety in df_preprocessed['Variety'].unique():
    # Find first week where rolling stats are missing
    mask = (df_preprocessed['Variety'] == variety) & df_preprocessed['Rolling_Mean_4w'].isna()
    first_idx = df_preprocessed[mask].index.min() if not df_preprocessed[mask].empty else None

    if first_idx is not None:
        # For first week, set rolling mean to current price
        current_price = df_preprocessed.loc[first_idx, 'Weekly_Price']
        df_preprocessed.loc[first_idx, rolling_cols[0::2]] = current_price  # Set means
        df_preprocessed.loc[first_idx, rolling_cols[1::2]] = 0  # Set standard deviations to 0

# Forward fill remaining missing rolling stats within each vegetable group
# This assumes if a value is missing, it should be same as previous week
for variety in df_preprocessed['Variety'].unique():
    mask = df_preprocessed['Variety'] == variety
    df_preprocessed.loc[mask, rolling_cols] = df_preprocessed.loc[mask, rolling_cols].ffill()

print("\nMissing values after imputation:")
print(df_preprocessed.isnull().sum())

Columns with missing values: ['Price_Std', 'Price_Change', 'Price_Change_Pct', 'Rolling_Std_4w', 'Rolling_Std_13w', 'Rolling_Std_26w']

Missing values after imputation:
Year                0
Month               0
Week_Number         0
Start_Date          0
End_Date            0
Location            0
Variety             0
Weekly_Price        0
Price_Std           0
Days_In_Week        0
Month_Year          0
Price_Change        0
Price_Change_Pct    0
Rolling_Mean_4w     0
Rolling_Std_4w      6
Rolling_Mean_13w    0
Rolling_Std_13w     6
Rolling_Mean_26w    0
Rolling_Std_26w     6
Quarter             0
Season              0
Price_Band          0
Fuel_Price          0
dtype: int64


In [9]:
# Define function to cap extreme values
def handle_outliers_iqr(df, column, lower_percentile=1, upper_percentile=99):
    """Cap values below lower percentile and above upper percentile"""
    # Calculate boundaries
    lower_bound = df[column].quantile(lower_percentile/100)
    upper_bound = df[column].quantile(upper_percentile/100)

    # Replace values outside boundaries with boundary values
    df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
    df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])

    return df

# Apply to price-related columns
price_cols = ['Weekly_Price', 'Price_Std', 'Price_Change', 'Price_Change_Pct']

for col in price_cols:
    df_preprocessed = handle_outliers_iqr(df_preprocessed, col, 0.5, 99.5)
    print(f"Outlier handling applied to {col}")

# Apply to rolling statistics columns too
for col in rolling_cols:
    df_preprocessed = handle_outliers_iqr(df_preprocessed, col, 0.5, 99.5)

Outlier handling applied to Weekly_Price
Outlier handling applied to Price_Std
Outlier handling applied to Price_Change
Outlier handling applied to Price_Change_Pct


In [10]:
 # Save original categorical values for reference
df_original_cats = df_preprocessed.copy()

# Initialize dictionary to store encoders
label_encoders = {}

# List of categorical columns to encode
categorical_cols = ['Variety', 'Location', 'Season', 'Price_Band', 'Quarter']

for col in categorical_cols:
    if col in df_preprocessed.columns:
        # Create encoder
        le = LabelEncoder()

        # Prepare all possible values including 'Unknown' for future data
        unique_vals = list(df_preprocessed[col].astype(str).unique()) + ['Unknown']
        le.fit(unique_vals)

        # Transform column to numbers
        df_preprocessed[f'{col}_encoded'] = le.transform(df_preprocessed[col].astype(str))
        label_encoders[col] = le  # Save encoder for later use

        # Show some mappings
        print(f"{col} encoding (first 10 values):")
        for i, val in enumerate(le.classes_):
            if i < 10:
                print(f"  {val} -> {i}")

# Create binary flags for the 5 most common vegetables
top_varieties = df_preprocessed['Variety'].value_counts().nlargest(5).index.tolist()
for variety in top_varieties:
    safe_name = variety.replace(" ", "_")  # Replace spaces with underscores
    df_preprocessed[f'Is_{safe_name}'] = (df_preprocessed['Variety'] == variety).astype(int)
    print(f"Created binary flag for: {variety}")

# Remove original categorical columns (keeping only encoded versions)
df_preprocessed = df_preprocessed.drop(columns=categorical_cols)
print(f"\nRemoved original categorical columns: {categorical_cols}")

Variety encoding (first 10 values):
  BITTER GOURD -> 0
  BRINJALS -> 1
  CABBAGE -> 2
  CARROT -> 3
  PUMPKIN -> 4
  TOMATOES -> 5
  Unknown -> 6
Location encoding (first 10 values):
  Colombo -> 0
  Unknown -> 1
Season encoding (first 10 values):
  Autumn -> 0
  Spring -> 1
  Summer -> 2
  Unknown -> 3
  Winter -> 4
Price_Band encoding (first 10 values):
  High -> 0
  Low -> 1
  Medium-High -> 2
  Medium-Low -> 3
  Unknown -> 4
Quarter encoding (first 10 values):
  1 -> 0
  2 -> 1
  3 -> 2
  4 -> 3
  Unknown -> 4
Created binary flag for: BITTER GOURD
Created binary flag for: BRINJALS
Created binary flag for: PUMPKIN
Created binary flag for: TOMATOES
Created binary flag for: CABBAGE

Removed original categorical columns: ['Variety', 'Location', 'Season', 'Price_Band', 'Quarter']


In [11]:
# Target leakage happens when we use information that wouldn't be available
# at prediction time. Price_Band might be calculated FROM Weekly_Price.

if 'Price_Band_encoded' in df_preprocessed.columns:
    print("\nAnalyzing Price_Band for potential target leakage...")

    # Check if Price_Band is just a categorized version of Weekly_Price
    df_check = df_original_cats.copy()

    # Create our own price bands based on Weekly_Price quartiles
    price_bands_manual = pd.qcut(df_check['Weekly_Price'], q=4,
                                 labels=['Very Low', 'Low', 'Medium', 'High'])

    # Compare with original
    matches = (df_check['Price_Band'].astype(str) == price_bands_manual.astype(str)).mean()
    print(f"Original Price_Band matches quartile-based bands: {matches:.2%}")

    if matches > 0.8:  # If more than 80% match
        print("WARNING: Price_Band appears to be derived from Weekly_Price!")
        print("If predicting Weekly_Price, remove Price_Band_encoded to avoid leakage.")
        print("If predicting Price_Band, keep it as target variable.")

    # Let's remove Price_Band_encoded to be safe (assuming we're predicting price)
    if 'Price_Band_encoded' in df_preprocessed.columns:
        df_preprocessed = df_preprocessed.drop(columns=['Price_Band_encoded'])
        print("Removed Price_Band_encoded to prevent target leakage")


Analyzing Price_Band for potential target leakage...
Original Price_Band matches quartile-based bands: 24.76%
Removed Price_Band_encoded to prevent target leakage


In [12]:
# Convert to datetime and sort chronologically
df_preprocessed['Start_Date'] = pd.to_datetime(df_preprocessed['Start_Date'])
df_preprocessed = df_preprocessed.sort_values(['Variety_encoded', 'Start_Date']).reset_index(drop=True)
print("Data sorted by vegetable and date")

# Create lagged features - these use ONLY past information
lag_cols = ['Weekly_Price', 'Price_Change', 'Price_Change_Pct']

for col in lag_cols:
    # shift(1) means "previous week's value"
    df_preprocessed[f'{col}_lag1'] = df_preprocessed.groupby('Variety_encoded')[col].shift(1)
    df_preprocessed[f'{col}_lag2'] = df_preprocessed.groupby('Variety_encoded')[col].shift(2)
    df_preprocessed[f'{col}_lag3'] = df_preprocessed.groupby('Variety_encoded')[col].shift(3)
    print(f"Created lag features for {col}")

# Verify rolling statistics don't use future data
df_preprocessed['Manual_Rolling_Mean_4w'] = df_preprocessed.groupby('Variety_encoded')['Weekly_Price'].transform(
    lambda x: x.shift(1).rolling(window=4, min_periods=1).mean()
)

# Fill NaN values in lagged features (first weeks have no previous data)
for col in df_preprocessed.columns:
    if 'lag' in col:
        df_preprocessed[col] = df_preprocessed[col].fillna(0)
        print(f"Filled NaN in {col} with 0")

Data sorted by vegetable and date
Created lag features for Weekly_Price
Created lag features for Price_Change
Created lag features for Price_Change_Pct
Filled NaN in Weekly_Price_lag1 with 0
Filled NaN in Weekly_Price_lag2 with 0
Filled NaN in Weekly_Price_lag3 with 0
Filled NaN in Price_Change_lag1 with 0
Filled NaN in Price_Change_lag2 with 0
Filled NaN in Price_Change_lag3 with 0
Filled NaN in Price_Change_Pct_lag1 with 0
Filled NaN in Price_Change_Pct_lag2 with 0
Filled NaN in Price_Change_Pct_lag3 with 0


In [13]:
# Identify which numeric columns to scale
# We don't scale: encoded categories, binary flags, ID numbers, dates
numeric_cols_to_scale = []
for col in df_preprocessed.select_dtypes(include=[np.number]).columns:
    # Skip columns that shouldn't be scaled
    if not any(x in col for x in ['_encoded', 'Is_', 'Week', 'Year', 'Month', 'Quarter', 'Days']):
        if df_preprocessed[col].nunique() > 10:  # Only scale columns with many values
            numeric_cols_to_scale.append(col)

print(f"\nScaling {len(numeric_cols_to_scale)} columns: {numeric_cols_to_scale}")

# Initialize StandardScaler (subtracts mean, divides by standard deviation)
scaler = StandardScaler()

# Scale selected columns
scaled_features = scaler.fit_transform(df_preprocessed[numeric_cols_to_scale])

# Create new column names for scaled features
scaled_df = pd.DataFrame(scaled_features,
                        columns=[f'{col}_scaled' for col in numeric_cols_to_scale])

# Add scaled features to dataframe
df_scaled = pd.concat([df_preprocessed, scaled_df], axis=1)

print(f"Added scaled versions of {len(numeric_cols_to_scale)} columns")


Scaling 17 columns: ['Price_Std', 'Price_Change', 'Price_Change_Pct', 'Rolling_Mean_4w', 'Rolling_Std_4w', 'Rolling_Mean_13w', 'Rolling_Std_13w', 'Rolling_Mean_26w', 'Rolling_Std_26w', 'Fuel_Price', 'Price_Change_lag1', 'Price_Change_lag2', 'Price_Change_lag3', 'Price_Change_Pct_lag1', 'Price_Change_Pct_lag2', 'Price_Change_Pct_lag3', 'Manual_Rolling_Mean_4w']
Added scaled versions of 17 columns


In [14]:
# Start with scaled data
df_final = df_scaled.copy()

# ENGINEER NEW FEATURES:

# 1. Time-based cyclical features (for seasonality)
df_final['Day_of_Year'] = df_final['Start_Date'].dt.dayofyear
df_final['Week_of_Year'] = df_final['Start_Date'].dt.isocalendar().week

# Sin/Cos transformation for months (makes December close to January)
df_final['Month_Sin'] = np.sin(2 * np.pi * df_final['Month']/12)
df_final['Month_Cos'] = np.cos(2 * np.pi * df_final['Month']/12)
print("Created cyclical time features")

# 2. Ratio features (relationships between variables)
df_final['Price_to_Fuel_Ratio'] = df_final['Weekly_Price'] / df_final['Fuel_Price']
df_final['Price_to_Rolling_Mean_Ratio'] = df_final['Weekly_Price'] / df_final['Rolling_Mean_4w']
print("Created ratio features")

# 3. Clean up any problematic values
df_final.replace([np.inf, -np.inf], np.nan, inplace=True)  # Replace infinity
df_final.fillna(0, inplace=True)  # Fill any remaining NaN

# 4. Remove duplicate columns
df_final = df_final.loc[:, ~df_final.columns.duplicated()]

# 5. Save the final dataset
df_final.to_csv('retail_fuel_weekly_preprocessed.csv', index=False)
print(f"\n Final preprocessed dataset saved!")
print(f"   Original shape: {df.shape}")
print(f"   Final shape: {df_final.shape}")
print(f"   File: retail_fuel_weekly_preprocessed.csv")

Created cyclical time features
Created ratio features

 Final preprocessed dataset saved!
   Original shape: (7784, 23)
   Final shape: (7784, 60)
   File: retail_fuel_weekly_preprocessed.csv


In [16]:
print("FINAL VALIDATION")

# Check 1: No missing values
missing_total = df_final.isnull().sum().sum()
print(f" Missing values: {missing_total} (should be 0)")

# Check 2: No infinite values
inf_count = np.isinf(df_final.select_dtypes(include=[np.number])).sum().sum()
print(f" Infinite values: {inf_count} (should be 0)")

# Check 3: Data types are appropriate
print(f"\n Data types distribution:")
print(df_final.dtypes.value_counts())

# Check 4: Sample of processed data - FIXED VERSION
print(f"\n Sample of processed data (5 rows):")

# Find what columns are actually available
scaled_cols = [col for col in df_final.columns if '_scaled' in col]
encoded_cols = [col for col in df_final.columns if '_encoded' in col]
price_cols = [col for col in df_final.columns if 'price' in col.lower() or 'Price' in col]

print(f"Available scaled columns: {scaled_cols[:5]}...")
print(f"Available encoded columns: {encoded_cols[:5]}...")
print(f"Available price columns: {price_cols}")

# Create safe sample columns
safe_sample_cols = []

# Try to get scaled versions first, fall back to original
for base_col in ['Weekly_Price', 'Price_Change', 'Fuel_Price']:
    scaled_version = f'{base_col}_scaled'
    if scaled_version in df_final.columns:
        safe_sample_cols.append(scaled_version)
    elif base_col in df_final.columns:
        safe_sample_cols.append(base_col)

# Add encoded columns if available
for enc_col in ['Variety_encoded', 'Season_encoded']:
    if enc_col in df_final.columns:
        safe_sample_cols.append(enc_col)

# Add Month_Sin if available
if 'Month_Sin' in df_final.columns:
    safe_sample_cols.append('Month_Sin')

print(f"\nUsing columns: {safe_sample_cols}")
if safe_sample_cols:
    print(df_final[safe_sample_cols].head())
else:
    print("No suitable columns found. Showing first 5 columns:")
    print(df_final.iloc[:, :5].head())

# Check 5: Save preprocessing artifacts for future use
print(f"\n Preprocessing complete!")
print(f"  Shape: {df_final.shape}")
print(f"  Columns: {len(df_final.columns)}")

# Save the preprocessed data
df_final.to_csv('retail_fuel_weekly_preprocessed.csv', index=False)
print(f"  Saved: retail_fuel_weekly_preprocessed.csv")

# Only save label_encoders and scaler if they exist
try:
    if 'label_encoders' in locals() and label_encoders:
        with open('label_encoders.pkl', 'wb') as f:
            pickle.dump(label_encoders, f)
        print(f"  Saved: label_encoders.pkl")
except NameError:
    print("  Note: label_encoders not found")

try:
    if 'scaler' in locals() and scaler:
        with open('feature_scaler.pkl', 'wb') as f:
            pickle.dump(scaler, f)
        print(f"  Saved: feature_scaler.pkl")
except NameError:
    print("  Note: scaler not found")

FINAL VALIDATION
 Missing values: 0 (should be 0)
 Infinite values: 0 (should be 0)

 Data types distribution:
float64           42
int64             13
object             2
datetime64[ns]     1
int32              1
UInt32             1
Name: count, dtype: int64

 Sample of processed data (5 rows):
Available scaled columns: ['Price_Std_scaled', 'Price_Change_scaled', 'Price_Change_Pct_scaled', 'Rolling_Mean_4w_scaled', 'Rolling_Std_4w_scaled']...
Available encoded columns: ['Variety_encoded', 'Location_encoded', 'Season_encoded', 'Quarter_encoded']...
Available price columns: ['Weekly_Price', 'Price_Std', 'Price_Change', 'Price_Change_Pct', 'Fuel_Price', 'Weekly_Price_lag1', 'Weekly_Price_lag2', 'Weekly_Price_lag3', 'Price_Change_lag1', 'Price_Change_lag2', 'Price_Change_lag3', 'Price_Change_Pct_lag1', 'Price_Change_Pct_lag2', 'Price_Change_Pct_lag3', 'Price_Std_scaled', 'Price_Change_scaled', 'Price_Change_Pct_scaled', 'Fuel_Price_scaled', 'Price_Change_lag1_scaled', 'Price_Change_lag

In [17]:
!git add .
!git commit -m "Data preprocessing and generate weekly retail, fuel preprocessed dataset"

[Market-Price-Prediction 2963b0c] Data preprocessing and generate weekly retail, fuel preprocessed dataset
 3 files changed, 7785 insertions(+)
 create mode 100644 feature_scaler.pkl
 create mode 100644 label_encoders.pkl
 create mode 100644 retail_fuel_weekly_preprocessed.csv


In [18]:
!git push origin Market-Price-Prediction

Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (4/4), 1.82 MiB | 1.93 MiB/s, done.
Total 4 (delta 1), reused 1 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Amika1118/DSGP_Group_38.git
   0cf435f..2963b0c  Market-Price-Prediction -> Market-Price-Prediction
